# Phase 1 — Lesson 12: Tensor Operations
> Faithful study notes built directly from the lesson source. Concepts, code, diagrams, and exercises.

**Source:** ai-engineering-from-scratch -> Phase 1 (Math Foundations), Lesson 12: Tensor Operations
**Type:** Build | **Languages:** Python | **Time:** ~90 minutes
**Prerequisites:** Phase 1, Lessons 01, 02



# What You'll Learn

---

## Learning Objectives

- Implement a tensor class with shape, strides, reshape, transpose, and element-wise operations from scratch
- Apply broadcasting rules to operate on tensors of different shapes without copying data
- Write einsum expressions for dot products, matrix multiplications, outer products, and batched operations
- Trace the exact tensor shapes through every step of multi-head attention


# What Is a Tensor?

---

A tensor is a multi-dimensional array. The number of dimensions is the rank:

| Rank | Name | Shape example |
|------|------|---------------|
| 0 | Scalar | () |
| 1 | Vector | (3,) |
| 2 | Matrix | (2, 3) |
| 3+ | Higher-order | (2, 2, 2) — image batch |

Total elements = product of all sizes. A (2, 3, 4) tensor holds 24 elements.


In [ ]:
import numpy as npscalar = np.array(5)vector = np.array([1, 2, 3])matrix = np.array([[1, 2], [3, 4]])tensor3d = np.array([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])for name, t in [('Scalar', scalar), ('Vector', vector), ('Matrix', matrix), ('3D Tensor', tensor3d)]:    print(f"{name:12s} shape={t.shape}  ndim={t.ndim}  size={t.size}")

# Shape, Strides, and Memory Layout

---

**Shape:** size of each dimension.
**Strides:** elements to skip to move one position along each dimension.

For shape (2, 3) in row-major order:
```
data = [a, b, c, d, e, f]  # flat storage
shape = (2, 3)
strides = (3, 1)           # skip 3 for row, 1 for column
```


In [ ]:
import numpy as npt = np.array([[1, 2, 3], [4, 5, 6]])print(f"Shape: {t.shape}")print(f"Strides: {t.strides}")print(f"t[0, 2] = {t[0, 2]}")print(f"t[1, 0] = {t[1, 0]}")# Index [i, j] -> flat index = i*strides[0] + j*strides[1]print(f"Flat index of [0, 2]: {0*3 + 2*1} = 2 -> {t.flat[2]}")print(f"Flat index of [1, 0]: {1*3 + 0*1} = 3 -> {t.flat[3]}")

In [ ]:
import numpy as npimport matplotlib.pyplot as plt# Visualize strides: transpose is O(1) with stride swapt = np.arange(6).reshape(2, 3)print(f"Original: shape={t.shape}, strides={t.strides}")print(f"Transposed: shape={t.T.shape}, strides={t.T.strides}")# No data copied in transposeprint(f"Same data buffer: {np.shares_memory(t, t.T)}")

# Broadcasting

---

Broadcasting lets you operate on tensors of different shapes without copying data.

```
Tensor A:     (8, 1, 6, 1)
Tensor B:        (7, 1, 5)
Padded B:     (1, 7, 1, 5)
Result:       (8, 7, 6, 5)
```


In [ ]:
import numpy as npimport matplotlib.pyplot as pltA = np.ones((8, 1, 6, 1))B = np.ones((7, 1, 5))C = A + Bprint(f"A shape: {A.shape}")print(f"B shape: {B.shape}")print(f"A + B shape: {C.shape}")

In [ ]:
import numpy as np# Pairwise distance via broadcastingX = np.random.randn(5, 2)Y = np.random.randn(4, 2)# X: (5, 2) -> (5, 1, 2), Y: (4, 2) -> (1, 4, 2)diff = X[:, None, :] - Y[None, :, :]dist = np.sqrt(np.sum(diff ** 2, axis=-1))print(f"X shape: {X.shape}, Y shape: {Y.shape}")print(f"Pairwise distance shape: {dist.shape}")print(f"Distance matrix:\n{np.round(dist, 2)}")

# Einsum: The Universal Tensor Operation

---

Einstein summation labels each axis with a letter. Axes in input but not output get summed.

| Operation | Einsum |
|-----------|--------|
| Dot product | "i,i->" |
| Outer product | "i,j->ij" |
| Matrix multiply | "ij,jk->ik" |
| Transpose | "ij->ji" |
| Batch matmul | "bij,bjk->bik" |
| Attention scores | "bhtd,bhsd->bhts" |


In [ ]:
import numpy as npa = np.array([1.0, 2.0, 3.0])b = np.array([4.0, 5.0, 6.0])print(f"Dot product:   {np.einsum('i,i->', a, b)}")print(f"Outer product:\n{np.einsum('i,j->ij', a, b)}")A = np.array([[1, 2], [3, 4]], dtype=float)B = np.array([[5, 6], [7, 8]], dtype=float)print(f"Matrix multiply:\n{np.einsum('ij,jk->ik', A, B)}")

In [ ]:
import numpy as npimport matplotlib.pyplot as plt# Multi-head attention via einsumB, H, T, D = 2, 4, 8, 16E = H * Dnp.random.seed(42)X = np.random.randn(B, T, E)W_q = np.random.randn(E, E) * 0.02W_k = np.random.randn(E, E) * 0.02W_v = np.random.randn(E, E) * 0.02Q = np.einsum('bte,ek->btk', X, W_q).reshape(B, T, H, D).transpose(0, 2, 1, 3)K = np.einsum('bte,ek->btk', X, W_k).reshape(B, T, H, D).transpose(0, 2, 1, 3)V = np.einsum('bte,ek->btk', X, W_v).reshape(B, T, H, D).transpose(0, 2, 1, 3)scores = np.einsum('bhtd,bhsd->bhts', Q, K) / np.sqrt(D)weights = np.exp(scores - scores.max(axis=-1, keepdims=True))weights = weights / weights.sum(axis=-1, keepdims=True)attn = np.einsum('bhts,bhsd->bhtd', weights, V)print(f"Input X shape: {X.shape}")print(f"Q/K/V shape: {Q.shape}")print(f"Scores shape: {scores.shape}")print(f"Attention output shape: {attn.shape}")print(f"\nEach step is a tensor op: project, split heads, score, weight, merge.")

# Quiz Review

---

Try answering these before reading the explanations:

**Q1:** What is a tensor's rank?
> **Explanation:** Number of dimensions (0D=scalar, 1D=vector, 2D=matrix).

**Q2:** What do strides enable?
> **Explanation:** O(1) transpose and reshape without copying data.

**Q3:** What does einsum 'ij,jk->ik' compute?
> **Explanation:** Matrix multiplication — sum over the shared j axis.



# Practice Exercises

---

Do these in order:

**Exercise 1:** Implement matrix multiplication using einsum and verify against @ operator.

**Exercise 2:** Implement broadcasting from scratch for shapes (3, 1) and (1, 4) producing (3, 4).

**Exercise 3:** Trace shapes through multi-head attention: input → Q/K/V → scores → weighted sum → output.



# Key Terms

---

| Term | What people say | What it actually means |
|------|----------------|----------------------|
| Tensor | A matrix but more dimensions | Multi-dimensional array with uniform type, shape, and strides |
| Rank | Number of dimensions | A matrix has rank 2 (not matrix rank) |
| Strides | How memory is laid out | Elements to skip per axis for indexing |
| Broadcasting | It just works when shapes differ | Align from right; dims must be equal or one must be 1 |
| Einsum | A fancy way to write matmul | General notation for any tensor contraction |


# Further Reading

---

- [NumPy Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)
- [einops](https://github.com/arogozhnikov/einops) — readable tensor reshaping
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)


# Next Steps

---

When you're ready, continue to Lesson 13: Numerical Stability.

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
